# 🏛️ PV Schaerbeek — pipeline d'extraction (v3, sécurisé & économe)

Notebook réorganisé **par phases**. Règle d'or : **on paie le LLM le moins possible**.

| Phase | Coût | Quand l'exécuter |
|---|---|---|
| 0 · Setup | gratuit | à chaque session |
| 1 · Audit complétude | **gratuit** (0 appel LLM) | à chaque session, en 1er |
| 2 · Re-extraction **ciblée** | faible (≈ trous réels) | **le mode normal** |
| 3 · Pipeline complet | **cher** | ⚠️ manuel, exceptionnel |
| 4 · Valider + copier le JSON | gratuit | avant de committer |
| 5 · Commit → `main` | gratuit | quand la base est bonne |
| 6 · Indexation Pinecone | quota embeddings | **quand tu auras du crédit** |

### 🔐 Secrets — AUCUNE clé en dur
Toutes les clés viennent du **gestionnaire de secrets Colab** (icône 🔑 à gauche).
À créer une seule fois, avec *Accès au notebook* activé :
- `ANTHROPIC_API_KEY`
- `GITHUB_TOKEN`
- `PINECONE_API_KEY`

> ⚠️ Les anciennes clé Anthropic et token GitHub qui étaient écrits en dur dans
> la v2 sont **compromises** : révoque-les et régénère-les avant d'aller plus loin.

## Phase 0 — Setup

In [ ]:
# 0.1 — Dépendances
!pip install -q requests beautifulsoup4 tqdm pdfplumber anthropic

In [ ]:
# 0.2 — Drive + dépôt calé EXACTEMENT sur origin/main (aucune fusion, pas de divergence)
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.isdir('/content/pv-explorer-app'):
    !git clone https://github.com/pmeyssonnier/pv-explorer-app.git /content/pv-explorer-app
%cd /content/pv-explorer-app
!git fetch origin main
!git checkout -f -B main origin/main

# État de la base versionnée dans le dépôt
!python -c "import json; d=json.load(open('backend/pv_conseil_schaerbeek.json')); print(len(d['seances']),'séances,', sum(len(s['points']) for s in d['seances']),'points')"

In [ ]:
# 0.3 — 🔐 Secrets depuis Colab (jamais écrits dans le notebook)
# Helper TOLÉRANT AU NOM : essaie plusieurs noms possibles du secret, prend
# le 1er trouvé. Repli getpass (saisie masquée) hors-Colab ou si aucun trouvé.
import os

def get_secret(*names, prompt=None):
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        for n in names:
            try:
                return userdata.get(n)
            except Exception:
                continue   # nom absent / accès non accordé → essaie le suivant
    import getpass
    return getpass.getpass(prompt or f'{names[0]} : ')

os.environ['ANTHROPIC_API_KEY'] = get_secret(
    'ANTHROPIC_API_KEY', 'anthropic_api_key', 'ANTHROPIC_KEY',
    prompt='Clé Anthropic : ')
print('✅ clé Anthropic chargée (len =', len(os.environ['ANTHROPIC_API_KEY']), ')')

In [ ]:
# 0.4 — Imports pipeline + garde-fous
%cd /content/pv-explorer-app/pipeline

# ⚠️ Purge le cache d'imports : force la relecture des .py FRAÎCHEMENT checkout
# (sinon Python garderait la version importée lors d'un run précédent, malgré le
#  git checkout de la Phase 0.2). Garantit que les imports = code de main.
import sys
for _m in ['pv_extraction_pipeline', 'audit_completeness', 'reextract_targeted']:
    sys.modules.pop(_m, None)

from pv_extraction_pipeline import (
    CONFIG, run_pipeline, validate_database, stats_summary, export_csv, get_pdf_list
)

print('MODEL      =', CONFIG['MODEL'])
print('MAX_TOKENS =', CONFIG['MAX_TOKENS'], '| CHUNK_SIZE =', CONFIG['CHUNK_SIZE'])
assert CONFIG['MAX_TOKENS'] == 8192 and CONFIG['CHUNK_SIZE'] == 8, \
    '❌ Ancien code en mémoire → Exécution > Redémarrer la session, puis relance Phase 0'

pdfs = get_pdf_list(CONFIG['INPUT_DIR'])
print(f'✅ {len(pdfs)} PDF prêts dans {CONFIG["INPUT_DIR"]}')

In [ ]:
# 0.4bis — ⚠️ Resynchronise Drive AVEC le dépôt, jamais l'inverse
#
# CONFIG['DB_JSON_PATH'] pointe vers Drive (persistant d'une session à
# l'autre, protège un run en cours d'un crash Colab) — PAS vers le dépôt
# fraîchement cloné en 0.2. Les deux copies divergent silencieusement dès
# qu'une correction passe par GitHub sans repasser par ce notebook (PR,
# panneau admin) : Drive reste alors sur un instantané ancien.
#
# Incident vécu : la Phase 2 a chargé une copie Drive vieille de plusieurs
# mois (1 284 points et 26 séances en moins, trois migrations de schéma
# absentes), l'a réextraite, et la Phase 5 l'a commitée PAR-DESSUS le bon
# état du dépôt — effaçant du travail que ce notebook n'avait jamais touché.
#
# Donc, à CHAQUE session, avant toute lecture (Phase 1) : Drive s'aligne sur
# le dépôt qu'on vient de cloner en 0.2 — jamais l'inverse. Un run interrompu
# reste protégé (c'est le rôle de BACKUP_DIR, séparé) ; ce qui ne doit
# JAMAIS arriver, c'est que Phase 1/2 lisent une base plus vieille que GitHub.
import os
import shutil

os.makedirs(CONFIG["DRIVE_ROOT"], exist_ok=True)
shutil.copy2("/content/pv-explorer-app/backend/pv_conseil_schaerbeek.json", CONFIG["DB_JSON_PATH"])
print(f"✅ Drive synchronisé sur le dépôt ({CONFIG['DB_JSON_PATH']})")

In [ ]:
# 0.5 — (optionnel, gratuit) vérifie que 2 PDF s'ouvrent, SANS appeler l'API
run_pipeline(max_files=2, dry_run=True)

## Phase 1 — Audit de complétude (GRATUIT, 0 appel LLM)

⚠️ Suppose la cellule **0.4bis** exécutée juste avant (Drive resynchronisé sur le dépôt) — sinon cette phase et la suivante travaillent sur une copie de la base potentiellement ancienne, sans le moindre avertissement.

Compare, séance par séance, les points **attendus** (regex `SP n.-` sur le PDF)
aux points **présents** dans la base. Révèle les trous **avant** de payer quoi que ce soit.

In [ ]:
from pv_extraction_pipeline import load_database, CONFIG
from audit_completeness import audit_completeness, print_audit

db = load_database()
report = audit_completeness(db, CONFIG['INPUT_DIR'])   # aucun appel Claude
summary = print_audit(report)                          # séances incomplètes + SP manqués

## Phase 2 — Re-extraction CIBLÉE  ✅ *mode normal*

Ne relance le LLM **que** sur les séances à trous réels (et, en interne, seulement
sur les **pages manquantes** via la récupération page-par-page). `min_missing=2`
ignore les off-by-one (« 87/88 » = souvent un faux positif regex, pas un vrai manque).

👉 C'est cette phase qu'on exécute au quotidien — **pas** la Phase 3.

In [ ]:
from reextract_targeted import targets_from_audit, reextract_seances

cibles = targets_from_audit(report, min_missing=2)   # ne garde que les vrais trous
print('Cibles :', cibles)

reextract_seances(cibles)    # re-extrait + fusionne + sauvegarde (n'écrase rien d'autre)

# Re-audit pour confirmer que les trous sont comblés
report2 = audit_completeness(load_database(), CONFIG['INPUT_DIR'])
print_audit(report2)

In [ ]:
# (optionnel) cibler manuellement 1-2 séances précises, par date ou nom de PDF
# reextract_seances(['2016-11-30', '2017-04-26'])   # les 2 plus gros gains

## Phase 3 — Pipeline complet  ⚠️ *CHER — ne pas exécuter par défaut*

À n'utiliser **que** pour un premier remplissage massif ou une refonte totale.
Ces cellules sont volontairement **désactivées** (code en commentaire) pour éviter
une facture inutile alors que la base est déjà à ~99,7 % complète.

- `run_pipeline()` : scanne **tous** les PDF (le cache SHA-256 + `progress.json`
  protègent, mais ça reste à éviter en routine).
- `run_pipeline(year=Y, force_reprocess=True)` : **ignore le cache** et re-facture
  **toute** l'année Y. Réserve-le à un PDF précis qu'on sait mal extrait.

> Pour combler des trous, préfère **toujours** la Phase 2 (ciblée).

In [ ]:
# ⚠️ DÉSACTIVÉ. Décommente en pleine conscience du coût (reprend via progress.json).
# db = run_pipeline()

In [ ]:
# ⚠️ DÉSACTIVÉ. force_reprocess=True IGNORE le cache → re-facture toute l'année.
# db = run_pipeline(year=2024, force_reprocess=True)

## Phase 4 — Valider et copier le JSON dans le dépôt

In [ ]:
from pv_extraction_pipeline import load_database, CONFIG
db = load_database()          # recharge la base réellement écrite sur le Drive

validate_database(db)         # titres manquants / SP dupliqués / montants suspects
stats_summary(db)
export_csv(db, '/content/drive/MyDrive/PV_Schaerbeek/pv_all_points.csv')

# Copie le JSON dans le dépôt cloné pour le committer
import shutil
SRC = CONFIG['DB_JSON_PATH']
DST = '/content/pv-explorer-app/backend/pv_conseil_schaerbeek.json'
shutil.copy(SRC, DST)
print('copié →', DST)

In [ ]:
# Contrôle final des compteurs avant commit
%cd /content/pv-explorer-app
!python -c "import json; d=json.load(open('backend/pv_conseil_schaerbeek.json')); print(len(d['seances']),'séances,', sum(len(s['points']) for s in d['seances']),'points')"

## Phase 5 — Commit du JSON → `main`

Le token vient des **secrets Colab** (`GITHUB_TOKEN`). Il est injecté dans l'URL du remote
*le temps du push*, puis **immédiatement retiré** du remote → jamais persistant, jamais
écrit dans le notebook.

In [ ]:
%cd /content/pv-explorer-app

# Rester calé sur main sans perdre le JSON qu'on vient de copier
!git fetch origin main
!git stash -u    # met de côté le JSON modifié
!git checkout -f -B main origin/main
!git stash pop   # remet le JSON par-dessus la base à jour

In [ ]:
# get_secret (défini en Phase 0.3) essaie plusieurs noms : GITHUB_TOKEN,
# github_token, GITHUB_PAT… → aucun besoin de renommer ton secret.
TOKEN = get_secret('GITHUB_TOKEN', 'github_token', 'GITHUB_PAT', 'github_pat',
                   prompt='GitHub token : ')

!git config user.email 'pmeyssonnier@gmail.com'
!git config user.name  'pmeyssonnier'
!git add backend/pv_conseil_schaerbeek.json
!git commit -m "data: mise à jour extraction PV Schaerbeek (re-extraction ciblée)"

# push avec le token en clair UNIQUEMENT dans la commande, puis remote nettoyé
!git remote set-url origin https://{TOKEN}@github.com/pmeyssonnier/pv-explorer-app.git
!git push origin main
!git remote set-url origin https://github.com/pmeyssonnier/pv-explorer-app.git
del TOKEN
print('✅ JSON poussé sur main — le backend Render servira la base à jour au prochain déploiement.')

## Phase 6 — Indexation Pinecone  🔜 *quand tu auras du crédit*

L'indexation consomme le **quota mensuel d'embeddings** (5 M tokens sur le plan gratuit).
Tant que le quota est épuisé, l'indexation renvoie des `429 RESOURCE_EXHAUSTED` **et**
le chat `/ask` est indisponible (l'embedding de la question passe par le même quota).

**À exécuter seulement une fois le plan Pinecone rechargé/upgradé.**

Astuce coût, du moins cher au plus cher :
1. **Option 0 — `reindex_points.py --depuis <ref>`** : seulement les points dont le
   texte a changé (souvent une poignée). À préférer après une correction de données.
2. **Option A — `--only-year`** : une ou plusieurs années entières, quand une année
   vient d'être ajoutée ou ré-extraite.
3. **Option B — réindex complet** : reconstruction, exceptionnel.

Dans tous les cas les ID sont stables : l'upsert est idempotent, on ne re-paie jamais
les embeddings des points qu'on n'envoie pas.

In [ ]:
!pip install -q pinecone==9.1.0

In [ ]:
# Récupérer le dernier index_pv.py + clé Pinecone depuis les secrets
%cd /content/pv-explorer-app
!git fetch origin main
!git checkout -f -B main origin/main

import os
os.environ['PINECONE_API_KEY'] = get_secret(
    'PINECONE_API_KEY', 'pinecone_api_key', 'PINECONE_KEY',
    prompt='PINECONE_API_KEY (pcsk_...) : ')
print('✅ clé Pinecone chargée')

### Option 0 — réindexation **ciblée** (la moins chère) ⭐

Une correction de données ne touche parfois que quelques points. `reindex_points.py`
compare la base d'aujourd'hui à celle d'une révision passée et n'envoie que les
vecteurs dont le **texte vectorisé** a réellement changé — la liste est *calculée*,
jamais recopiée à la main.

Exemple : la correction des motions rejetées (`3fb3922`, PR #171) → **10 points**
au lieu des sept années entières qu'aurait ré-embeddées `--only-year`.

Toujours lancer le `--dry-run` d'abord : il relit l'index et dit, point par point,
si la mise à jour est encore nécessaire.

In [ ]:
# Option 0.a — ce qui SERAIT envoyé (aucune écriture, aucun embedding consommé)
%cd /content/pv-explorer-app/backend
!python reindex_points.py --depuis 3fb3922 --dry-run

In [ ]:
# Option 0.b — envoi réel, puis relecture de l'index pour confirmer
%cd /content/pv-explorer-app/backend
!python reindex_points.py --depuis 3fb3922 --verifier

In [ ]:
# Option A (économe) — n'indexer QUE les années manquantes
%cd /content/pv-explorer-app/backend
!python index_pv.py --input pv_conseil_schaerbeek.json --commune schaerbeek --only-year 2019,2021,2022,2023

In [ ]:
# Option B — réindex complet (⚠️ gros consommateur de quota d'embeddings)
# %cd /content/pv-explorer-app/backend
# !python index_pv.py --input pv_conseil_schaerbeek.json --commune schaerbeek